## Análisis de casos recoordinados
Resumen visual de volumen y costo operativo.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Etapa 1: Carga de datos
Leemos el dataset ya transformado desde processed.

In [ ]:
casos_recoordinados = pd.read_csv('../data/processed/casos_recoordinados.csv', index_col=False)

In [ ]:
casos_recoordinados

### Etapa 2: Visualización integrada
Construimos métricas por tipo, costo y hora en una sola figura.

In [ ]:
# Paleta solicitada
PALETTE = ['#ffdbea', '#d8f5f6', '#fff6d9', '#d8f7df']
PRIMARY = '#ff012b'

df = casos_recoordinados.copy()
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%y %H:%M:%S', errors='coerce')
df = df.dropna(subset=['fecha'])
df['hora'] = df['fecha'].dt.hour

# 1) Pie: distribucion de casos por tipo (cantidad)
type_counts = df['type'].value_counts().sort_values(ascending=False)

# 2) Barras: top tipos por suma de costo
top_tipos_costo = (
    df.groupby('type', as_index=False)['costo']
    .sum()
    .sort_values('costo', ascending=False)
    .head(10)
)

# 3) Histograma: costo acumulado por hora del dia

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'xy'}],
           [{'type': 'xy', 'colspan': 2}, None]],
    subplot_titles=(
        'Distribucion de casos por tipo',
        'Top tipos por costo total',
        'Costo operativo por hora del dia'
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.08
)

# Pie
pie_colors = [PRIMARY] + PALETTE * 10
fig.add_trace(
    go.Pie(
        labels=type_counts.index,
        values=type_counts.values,
        marker=dict(colors=pie_colors[:len(type_counts)]),
        textinfo='label+percent',
        hole=0.35
    ),
    row=1, col=1
)

# Barras top por costo total de tipo
bar_colors = [PRIMARY] + PALETTE * 10
fig.add_trace(
    go.Bar(
        x=top_tipos_costo['type'],
        y=top_tipos_costo['costo'],
        marker_color=bar_colors[:len(top_tipos_costo)],
        hovertemplate='Tipo: %{x}<br>Costo total: %{y:$,.0f}<extra></extra>'
    ),
    row=1, col=2
)

# Histograma costo por hora
fig.add_trace(
    go.Histogram(
        x=df['hora'],
        y=df['costo'],
        histfunc='sum',
        xbins=dict(start=0, end=24, size=1),
        marker=dict(color=PRIMARY, line=dict(color='#ffffff', width=1)),
        hovertemplate='Hora %{x}:00<br>Costo total: %{y:$,.0f}<extra></extra>'
    ),
    row=2, col=1
)

fig.update_layout(
    title='Costos operativos de casos recoordinados',
    template='plotly_white',
    height=850,
    showlegend=False,
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff'
    )

fig.update_xaxes(title_text='Tipo de caso', tickangle=-25, row=1, col=2)
fig.update_yaxes(title_text='Costo operativo total', tickprefix='$', row=1, col=2)
fig.update_xaxes(title_text='Hora del dia', dtick=1, row=2, col=1)
fig.update_yaxes(title_text='Costo total', tickprefix='$', row=2, col=1)

fig.show()